# LeiGS 2026 — Proctor Prediction Challenge
**Goal**: Predict `proctor_mdd_g_cm3` (MDD) and `proctor_owc_pct` (OWC) from soil classification features.
**Metric**: NMAE = 0.5×(MAE_mdd/IQR_mdd) + 0.5×(MAE_owc/IQR_owc)

## Section 0: Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
np.random.seed(42)

# IQR values from full training set (fixed — do not recompute per fold)
IQR_MDD = 0.198
IQR_OWC = 3.860
RHO_W = 1.0

print('All imports OK')

## Section 1: Data Loading

In [ ]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
print(f'Test IDs: {test_raw["id"].min()} – {test_raw["id"].max()}')
train_raw.head(3)

## Section 2: EDA

In [ ]:
TARGET_COLS = ['proctor_mdd_g_cm3', 'proctor_owc_pct']

# Target distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, TARGET_COLS):
    ax.hist(train_raw[col], bins=25, edgecolor='k', color='steelblue', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel(col)
    q1, q3 = train_raw[col].quantile(0.25), train_raw[col].quantile(0.75)
    ax.axvline(q1, color='orange', linestyle='--', label=f'Q1={q1:.3f}')
    ax.axvline(q3, color='red',    linestyle='--', label=f'Q3={q3:.3f}')
    ax.legend()
plt.suptitle('Target Variable Distributions', fontsize=13)
plt.tight_layout()
plt.show()
print(train_raw[TARGET_COLS].describe().round(3))

In [ ]:
# Missing value heatmap
missing = train_raw.isnull().mean() * 100
missing_cols = missing[missing > 0].sort_values(ascending=False)
print('Missing value rates (%):')
print(missing_cols.round(1))

fig, ax = plt.subplots(figsize=(8, 3))
missing_cols.plot.bar(ax=ax, color='coral', edgecolor='k')
ax.set_ylabel('% Missing')
ax.set_title('Missing Values by Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Key scatter plots: fines% vs targets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target in zip(axes, TARGET_COLS):
    ax.scatter(train_raw['psd_passing_at_0_063mm_pct'], train_raw[target],
               alpha=0.6, edgecolors='k', linewidths=0.3, color='steelblue')
    ax.set_xlabel('Fines fraction < 0.063 mm (%)')
    ax.set_ylabel(target)
    corr = train_raw[['psd_passing_at_0_063mm_pct', target]].corr().iloc[0, 1]
    ax.set_title(f'{target}  (r={corr:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# Proctor mold diameter distribution
print('Proctor mold diameter distribution:')
print(train_raw['proctor_diam_mm'].value_counts())

# Grain density distribution
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(train_raw['grain_density_g_cm3'], bins=20, edgecolor='k', color='seagreen', alpha=0.8)
ax.set_xlabel('Grain density (g/cm³)')
ax.set_title('Grain Density Distribution (should be ~2.65 for quartzitic soils)')
plt.tight_layout()
plt.show()

## Section 3: Preprocessing

In [ ]:
PSD_D_COLS = [
    'psd_size_at_d10_mm', 'psd_size_at_d20_mm', 'psd_size_at_d30_mm',
    'psd_size_at_d40_mm', 'psd_size_at_d50_mm', 'psd_size_at_d60_mm',
    'psd_size_at_d70_mm', 'psd_size_at_d80_mm', 'psd_size_at_d90_mm',
    'psd_size_at_d95_mm', 'psd_size_at_d98_mm'
]

HIGH_MISSING = [
    'atterberg_liquid_limit_pct', 'atterberg_plastic_limit_pct',
    'hyd_cond_kf_m_s', 'hyd_cond_hyd_gradient', 'loss_on_ignition_pct'
]

def preprocess(df, medians=None, fit=False):
    df = df.copy()

    # Boolean coercion
    df['psd_has_sedimentation'] = df['psd_has_sedimentation'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0}
    ).astype(float)

    # Numeric coercion
    for col in PSD_D_COLS + HIGH_MISSING + [
        'psd_passing_at_0_002mm_pct', 'psd_passing_at_0_063mm_pct',
        'psd_passing_at_2mm_pct', 'grain_density_g_cm3',
        'proctor_diam_mm'
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Missingness indicator flags (fit on train, apply to test)
    for col in HIGH_MISSING:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # Median imputation — fit on train, transform both
    if fit:
        medians = {col: df[col].median() for col in HIGH_MISSING}
    for col in HIGH_MISSING:
        df[col] = df[col].fillna(medians[col])

    return df, medians

train_pp, medians = preprocess(train_raw, fit=True)
test_pp, _        = preprocess(test_raw, medians=medians, fit=False)

print('Preprocessing complete.')
print(f'Remaining NaNs in train: {train_pp.drop(columns=TARGET_COLS, errors="ignore").isnull().sum().sum()}')
print(f'Remaining NaNs in test:  {test_pp.isnull().sum().sum()}')

## Section 4: Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()

    D10 = df['psd_size_at_d10_mm'].replace(0, np.nan)
    D30 = df['psd_size_at_d30_mm']
    D60 = df['psd_size_at_d60_mm'].replace(0, np.nan)

    # Geotechnical gradation indices
    df['feat_cu'] = D60 / D10                           # Coefficient of uniformity
    df['feat_cc'] = (D30 ** 2) / (D60 * D10)           # Coefficient of curvature
    df['feat_log_cu'] = np.log1p(df['feat_cu'])        # Log-CU (skewed distribution)

    # Plasticity Index
    df['feat_pi'] = (
        df['atterberg_liquid_limit_pct'] - df['atterberg_plastic_limit_pct']
    )

    # Grain-size fractions
    df['feat_sand_pct']   = (df['psd_passing_at_2mm_pct']
                              - df['psd_passing_at_0_063mm_pct']).clip(lower=0)
    df['feat_gravel_pct'] = (100 - df['psd_passing_at_2mm_pct']).clip(lower=0)
    df['feat_clay_to_fines_ratio'] = (
        df['psd_passing_at_0_002mm_pct']
        / df['psd_passing_at_0_063mm_pct'].replace(0, np.nan)
    ).fillna(0)

    # Soil type classification (coarse=0, mixed=1, fine=2)
    df['feat_soil_type'] = pd.cut(
        df['psd_passing_at_0_063mm_pct'],
        bins=[-1, 10, 40, 101], labels=[0, 1, 2]
    ).astype(float)

    # Physics-informed: saturation line MDD at a rough wopt estimate
    # ρd_sat = ρs / (1 + w·ρs/ρw), use fines%/10 as rough wopt proxy
    rho_s = df['grain_density_g_cm3']
    w_est = (df['psd_passing_at_0_063mm_pct'] / 10).clip(lower=3)
    df['feat_sat_mdd_proxy'] = rho_s / (1 + (w_est / 100) * rho_s / RHO_W)

    # Median grain size
    df['feat_d50'] = df['psd_size_at_d50_mm']

    # Log-transform PSD D-cols (span orders of magnitude)
    for col in PSD_D_COLS:
        df[f'log_{col}'] = np.log1p(df[col])

    # Log hydraulic conductivity
    df['log_hyd_cond_kf_m_s'] = np.log1p(df['hyd_cond_kf_m_s'])

    return df

train_fe = engineer_features(train_pp)
test_fe  = engineer_features(test_pp)

# Feature column list (exclude id and targets)
EXCLUDE = ['id'] + TARGET_COLS
FEATURE_COLS = [c for c in train_fe.columns if c not in EXCLUDE]

print(f'Feature count: {len(FEATURE_COLS)}')
print('Features:', FEATURE_COLS[:10], '...')

X_train = train_fe[FEATURE_COLS].values.astype(np.float32)
y_mdd   = train_fe['proctor_mdd_g_cm3'].values
y_owc   = train_fe['proctor_owc_pct'].values
X_test  = test_fe[FEATURE_COLS].values.astype(np.float32)

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'NaN in X_train: {np.isnan(X_train).sum()}  X_test: {np.isnan(X_test).sum()}')

## Section 5: Metric & Baseline Model

In [ ]:
def nmae(y_mdd_true, y_owc_true, y_mdd_pred, y_owc_pred):
    mae_mdd = np.mean(np.abs(y_mdd_true - y_mdd_pred))
    mae_owc = np.mean(np.abs(y_owc_true - y_owc_pred))
    return 0.5 * (mae_mdd / IQR_MDD) + 0.5 * (mae_owc / IQR_OWC)

def cv_nmae(model_mdd, model_owc, X, y_mdd, y_owc, n_splits=5):
    """Run n-fold CV and return per-fold NMAE scores."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_mdd_tr, y_mdd_val = y_mdd[tr_idx], y_mdd[val_idx]
        y_owc_tr, y_owc_val = y_owc[tr_idx], y_owc[val_idx]

        model_mdd.fit(X_tr, y_mdd_tr)
        model_owc.fit(X_tr, y_owc_tr)

        pred_mdd = model_mdd.predict(X_val)
        pred_owc = model_owc.predict(X_val)

        score = nmae(y_mdd_val, y_owc_val, pred_mdd, pred_owc)
        scores.append(score)
        print(f'  Fold {fold+1}: NMAE = {score:.4f}')

    mean_score = np.mean(scores)
    std_score  = np.std(scores)
    print(f'  Mean NMAE: {mean_score:.4f} ± {std_score:.4f}')
    return scores

print('--- Baseline: Ridge Regression ---')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

baseline_mdd = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
baseline_owc = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])

baseline_scores = cv_nmae(baseline_mdd, baseline_owc, X_train, y_mdd, y_owc)

## Section 6: XGBoost & LightGBM

In [ ]:
print('--- XGBoost ---')
xgb_mdd = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_owc = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_scores = cv_nmae(xgb_mdd, xgb_owc, X_train, y_mdd, y_owc)

In [ ]:
print('--- LightGBM ---')
lgb_mdd = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_owc = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_scores = cv_nmae(lgb_mdd, lgb_owc, X_train, y_mdd, y_owc)

In [ ]:
# Feature importance (fit on full train)
xgb_mdd.fit(X_train, y_mdd)
xgb_owc.fit(X_train, y_owc)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, model, title in zip(axes, [xgb_mdd, xgb_owc], ['XGB: MDD importances', 'XGB: OWC importances']):
    imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)[:20]
    imp.plot.bar(ax=ax, color='steelblue', edgecolor='k')
    ax.set_title(title)
    ax.tick_params(axis='x', labelsize=7)
plt.tight_layout()
plt.show()

## Section 7: Ensemble

In [ ]:
def cv_nmae_ensemble(w_xgb, X, y_mdd, y_owc, n_splits=5):
    """CV for weighted ensemble of XGB + LGB."""
    w_lgb = 1.0 - w_xgb
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_mdd_tr, y_mdd_val = y_mdd[tr_idx], y_mdd[val_idx]
        y_owc_tr, y_owc_val = y_owc[tr_idx], y_owc[val_idx]

        # Fit fresh copies
        _xm = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                                reg_lambda=1.0, random_state=42, n_jobs=-1, verbosity=0)
        _xo = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                                reg_lambda=1.0, random_state=42, n_jobs=-1, verbosity=0)
        _lm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31,
                                 subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                                 reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1)
        _lo = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31,
                                 subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                                 reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1)

        _xm.fit(X_tr, y_mdd_tr); _xo.fit(X_tr, y_owc_tr)
        _lm.fit(X_tr, y_mdd_tr); _lo.fit(X_tr, y_owc_tr)

        pred_mdd = w_xgb * _xm.predict(X_val) + w_lgb * _lm.predict(X_val)
        pred_owc = w_xgb * _xo.predict(X_val) + w_lgb * _lo.predict(X_val)

        scores.append(nmae(y_mdd_val, y_owc_val, pred_mdd, pred_owc))
    return np.mean(scores)

# Grid-search blend weight
print('Searching best XGB blend weight...')
weights = np.arange(0.1, 1.0, 0.1)
blend_scores = {w: cv_nmae_ensemble(w, X_train, y_mdd, y_owc) for w in weights}
best_w = min(blend_scores, key=blend_scores.get)
print(f'\nBlend scores: {[(round(w,1), round(s,4)) for w, s in blend_scores.items()]}')
print(f'Best XGB weight: {best_w:.1f}  →  NMAE: {blend_scores[best_w]:.4f}')

## Section 8: Hyperparameter Tuning (Optuna)

In [ ]:
def make_objective(target_y, model_cls):
    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 200, 800),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            max_depth        = trial.suggest_int('max_depth', 3, 7),
            subsample        = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha        = trial.suggest_float('reg_alpha', 0.0, 2.0),
            reg_lambda       = trial.suggest_float('reg_lambda', 0.0, 2.0),
            random_state=42, n_jobs=-1,
        )
        if model_cls == xgb.XGBRegressor:
            params['verbosity'] = 0
        else:
            params['verbose'] = -1
            params['num_leaves'] = trial.suggest_int('num_leaves', 15, 63)

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for tr_idx, val_idx in kf.split(X_train):
            m = model_cls(**params)
            m.fit(X_train[tr_idx], target_y[tr_idx])
            scores.append(np.mean(np.abs(target_y[val_idx] - m.predict(X_train[val_idx]))))
        return np.mean(scores)  # minimize MAE per target
    return objective

N_TRIALS = 50  # increase to 100+ for best results (slower)

print(f'Tuning XGBoost for MDD ({N_TRIALS} trials)...')
study_xgb_mdd = optuna.create_study(direction='minimize')
study_xgb_mdd.optimize(make_objective(y_mdd, xgb.XGBRegressor), n_trials=N_TRIALS)
best_xgb_mdd_params = study_xgb_mdd.best_params
print(f'Best MAE_mdd: {study_xgb_mdd.best_value:.4f}')

print(f'\nTuning XGBoost for OWC ({N_TRIALS} trials)...')
study_xgb_owc = optuna.create_study(direction='minimize')
study_xgb_owc.optimize(make_objective(y_owc, xgb.XGBRegressor), n_trials=N_TRIALS)
best_xgb_owc_params = study_xgb_owc.best_params
print(f'Best MAE_owc: {study_xgb_owc.best_value:.4f}')

print(f'\nTuning LightGBM for MDD ({N_TRIALS} trials)...')
study_lgb_mdd = optuna.create_study(direction='minimize')
study_lgb_mdd.optimize(make_objective(y_mdd, lgb.LGBMRegressor), n_trials=N_TRIALS)
best_lgb_mdd_params = study_lgb_mdd.best_params
print(f'Best MAE_mdd: {study_lgb_mdd.best_value:.4f}')

print(f'\nTuning LightGBM for OWC ({N_TRIALS} trials)...')
study_lgb_owc = optuna.create_study(direction='minimize')
study_lgb_owc.optimize(make_objective(y_owc, lgb.LGBMRegressor), n_trials=N_TRIALS)
best_lgb_owc_params = study_lgb_owc.best_params
print(f'Best MAE_owc: {study_lgb_owc.best_value:.4f}')

In [ ]:
# Evaluate tuned models with NMAE
def clean_params(params, model_cls):
    """Remove model-specific params that don't belong."""
    p = dict(params)
    if model_cls == xgb.XGBRegressor:
        p.pop('num_leaves', None)
        p['verbosity'] = 0
    else:
        p['verbose'] = -1
    p['random_state'] = 42
    p['n_jobs'] = -1
    return p

tuned_xgb_mdd = xgb.XGBRegressor(**clean_params(best_xgb_mdd_params, xgb.XGBRegressor))
tuned_xgb_owc = xgb.XGBRegressor(**clean_params(best_xgb_owc_params, xgb.XGBRegressor))
tuned_lgb_mdd = lgb.LGBMRegressor(**clean_params(best_lgb_mdd_params, lgb.LGBMRegressor))
tuned_lgb_owc = lgb.LGBMRegressor(**clean_params(best_lgb_owc_params, lgb.LGBMRegressor))

print('--- Tuned XGBoost CV ---')
tuned_xgb_scores = cv_nmae(tuned_xgb_mdd, tuned_xgb_owc, X_train, y_mdd, y_owc)
print('--- Tuned LightGBM CV ---')
tuned_lgb_scores = cv_nmae(tuned_lgb_mdd, tuned_lgb_owc, X_train, y_mdd, y_owc)

In [ ]:
# Model comparison summary
comparison = pd.DataFrame({
    'Model': ['Baseline (Ridge)', 'XGBoost (default)', 'LightGBM (default)',
              f'Ensemble (w_xgb={best_w:.1f})',
              'XGBoost (tuned)', 'LightGBM (tuned)'],
    'CV NMAE (mean)': [
        np.mean(baseline_scores),
        np.mean(xgb_scores),
        np.mean(lgb_scores),
        blend_scores[best_w],
        np.mean(tuned_xgb_scores),
        np.mean(tuned_lgb_scores)
    ],
    'CV NMAE (std)': [
        np.std(baseline_scores),
        np.std(xgb_scores),
        np.std(lgb_scores),
        np.nan,
        np.std(tuned_xgb_scores),
        np.std(tuned_lgb_scores)
    ]
}).sort_values('CV NMAE (mean)')

print('=== Model Comparison ===')
print(comparison.round(4).to_string(index=False))

## Section 9: Final Prediction

In [ ]:
def apply_saturation_constraint(mdd_pred, owc_pred, grain_density, safety=0.99):
    """MDD must not exceed the zero-air-voids saturation line."""
    rho_s = grain_density
    w_frac = owc_pred / 100.0
    sat_limit = rho_s / (1 + w_frac * rho_s / RHO_W)
    return np.minimum(mdd_pred, sat_limit * safety)

# Pick best model from comparison
best_model_name = comparison.iloc[0]['Model']
print(f'Best CV model: {best_model_name}  NMAE={comparison.iloc[0]["CV NMAE (mean)"]:.4f}')

# Fit winning models on full training set
# Default: use tuned ensemble (tune which models to use based on comparison above)
final_mdd = tuned_lgb_mdd if 'LightGBM' in best_model_name else tuned_xgb_mdd
final_owc = tuned_lgb_owc if 'LightGBM' in best_model_name else tuned_xgb_owc

# Re-init to avoid state from CV
final_mdd = final_mdd.__class__(**final_mdd.get_params())
final_owc = final_owc.__class__(**final_owc.get_params())

final_mdd.fit(X_train, y_mdd)
final_owc.fit(X_train, y_owc)

pred_mdd = final_mdd.predict(X_test)
pred_owc = final_owc.predict(X_test)

# Apply physical saturation constraint
grain_dens_test = test_fe['grain_density_g_cm3'].values
pred_mdd_constrained = apply_saturation_constraint(pred_mdd, pred_owc, grain_dens_test)

n_clipped = (pred_mdd_constrained < pred_mdd).sum()
print(f'Saturation constraint applied: {n_clipped} predictions clipped')
print(f'\nPrediction ranges:')
print(f'  MDD: [{pred_mdd_constrained.min():.3f}, {pred_mdd_constrained.max():.3f}]  (train: [1.609, 2.209])')
print(f'  OWC: [{pred_owc.min():.3f}, {pred_owc.max():.3f}]  (train: [3.54, 20.71])')

## Section 10: Submission Validation & Export

In [ ]:
submission = pd.DataFrame({
    'id': test_raw['id'],
    'proctor_owc_pct': pred_owc,
    'proctor_mdd_g_cm3': pred_mdd_constrained
})

# Validation assertions
assert list(submission.columns) == ['id', 'proctor_owc_pct', 'proctor_mdd_g_cm3'], 'Wrong columns'
assert len(submission) == len(test_raw), f'Row count mismatch: {len(submission)} vs {len(test_raw)}'
assert (submission['id'] == test_raw['id']).all(), 'ID mismatch with test.csv'
assert (submission['proctor_mdd_g_cm3'] > 1.4).all(), 'MDD below physical minimum'
assert (submission['proctor_mdd_g_cm3'] < 2.5).all(), 'MDD above physical maximum'
assert (submission['proctor_owc_pct'] > 2.0).all(),   'OWC below physical minimum'
assert (submission['proctor_owc_pct'] < 35.0).all(),  'OWC above physical maximum'

print('All validation checks passed!')
print(submission.describe().round(3))

# Save
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
out_path = f'submissions/submission_{timestamp}_v1.csv'
submission.to_csv(out_path, index=False)
print(f'\nSubmission saved: {out_path}')
submission.head(10)

In [ ]:
# Final diagnostic plot: predicted vs train distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (train_y, pred_y, col) in zip(axes, [
    (y_mdd, pred_mdd_constrained, 'proctor_mdd_g_cm3'),
    (y_owc, pred_owc, 'proctor_owc_pct')
]):
    ax.hist(train_y, bins=20, alpha=0.6, label='Train', color='steelblue', edgecolor='k')
    ax.hist(pred_y,  bins=20, alpha=0.6, label='Test predictions', color='coral', edgecolor='k')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Train distribution vs Test predictions (should overlap well)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Final CV NMAE Summary ===')
print(comparison.round(4).to_string(index=False))